# basic

In [2]:
%load_ext autoreload
%autoreload all

In [1]:
from openai import OpenAI
import json

BASE_URL = "http://localhost:11434/v1"
API_KEY = "ollama"
MODEL = "qwen2.5:7b"
LLM_OPTIONS = {"reasoning_effort": "none"} if API_KEY == "ollama" else {}
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


In [3]:
MAX_TURNS = 6

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Use the calculate tool whenever the user's task requires arithmetic.
Only give your final answer once you have all the information you need.
"""


# 1. tool definition

In [4]:
def calculate(expression: str) -> str:
    """
    Safely evaluate a basic arithmetic expression, e.g. "12 * (3 + 4)".

    We use Python's eval() here ONLY for teaching simplicity, restricted to
    a tiny safe namespace (no builtins). In a real production harness you
    would use a proper math expression parser instead of eval, because eval
    is dangerous with untrusted input even when restricted like this.
    """
    try:
        # `{"__builtins__": {}}` strips away dangerous built-in functions
        # (like open, import, etc.) so the eval is a bit safer, though still
        # not something you'd want to expose to fully untrusted users.
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        # Tools can fail! Returning the error as a string lets the LLM see
        # what went wrong and decide how to react (e.g. retry, ask user).
        return f"ERROR: could not evaluate '{expression}': {e}"
    

# 2. Describe the tool schema to the LLM using the OpenAI tool-calling schema.

In [5]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a basic arithmetic expression and return the numeric result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python-style arithmetic expression, e.g. '2 + 2' or '(5*3)/2'.",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]

# 3. tool registration

In [6]:
# Registry mapping tool name -> actual Python function to call.
# This pattern (a dict lookup) is what Module 3 will expand into a full
# "tool registry" system when we add many more tools.
TOOL_REGISTRY = {
    "calculate": calculate,
}


# call the tool in a agent loop

In [22]:
def run_agent_loop(user_task: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_task},
    ]

    for turn in range(1, MAX_TURNS + 1):
        print(f"\n{'=' * 60}")
        print(f"[STEP {turn}] Sending {len(messages)} messages to the LLM (tools enabled)...")
        print(f"{'=' * 60}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS_SCHEMA,
            temperature=0.0,
            **LLM_OPTIONS,
        )

        message = response.choices[0].message
    

        # Case 1: the model wants to call one or more tools.    
        if message.tool_calls:
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments) # this

                print(f"\n[STEP {turn}] Model wants to call tool '{tool_name}' with args: {tool_args}")

                if tool_name not in TOOL_REGISTRY:
                    raise ValueError(f"Tool '{tool_name}' is not registered.")

                # Call the actual Python function for the requested tool.
                tool_function = TOOL_REGISTRY[tool_name]
                tool_result = tool_function(**tool_args)

                print(f"[STEP {turn}] Tool '{tool_name}' returned: {tool_result}")

                # Add the model's tool call and the tool's result to the conversation.
                messages.append(message)
                messages.append({"role": "tool", 
                                 "tool_call_id": tool_call.id,
                                 "content": str(tool_result)})

        # Case 2: the model gave a normal text answer -- no tool call means
        # it considers itself done (this is the standard convention for
        # tool-calling APIs: absence of tool_calls == final answer).
        else:
            print(f"\n[STEP {turn}] Model gave a final answer (no tool calls).")
            messages.append(message)
            return message.content
        
    print("\n[WARNING] Max turns reached without a final answer.")
    return "(Agent did not finish within the turn limit.)"
    

In [24]:
if __name__ == "__main__":
    # A task that specifically requires arithmetic, so you can watch the
    # model choose to call the calculator tool instead of guessing the answer.
    task = "What is ((438 * 12) - 57)/ 3? Use tools if necessary. Give me just the final number"
    print(f"USER TASK: {task}")

    result = run_agent_loop(task)

    print(f"\n{'#' * 60}")
    print("FINAL ANSWER:")
    print(result)
    print(f"{'#' * 60}")

USER TASK: What is ((438 * 12) - 57)/ 3? Use tools if necessary. Give me just the final number

[STEP 1] Sending 2 messages to the LLM (tools enabled)...

[STEP 1] Model wants to call tool 'calculate' with args: {'expression': '((438 * 12) - 57) / 3'}
[STEP 1] Tool 'calculate' returned: 1733.0

[STEP 2] Sending 4 messages to the LLM (tools enabled)...

[STEP 2] Model gave a final answer (no tool calls).

############################################################
FINAL ANSWER:
The result of the expression ((438 * 12) - 57) / 3 is 1733.0.
############################################################
